# Task 5 (Day 5): Spot unusual attendance without labels
**Objective.** Build an anomaly detector for attendance, then measure how well it works against hidden labels.
**Why it matters.** Not every problem comes with answers; anomaly detection finds the odd one out and helps HR catch errors early.
**You will learn.** Unsupervised learning, IsolationForest, contamination, precision/recall for anomalies, mixing simple rules with a model.
**Rule.** Labels are for *measuring*, never for training. The flags are saved first; labels are read afterwards.

In [1]:
import sys, pathlib, inspect, subprocess
ROOT = pathlib.Path.cwd().resolve()
while not (ROOT / 'src').exists(): ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'src')); sys.path.insert(0, str(ROOT / '05_Starter'))
import numpy as np, pandas as pd, matplotlib.pyplot as plt
pd.set_option('display.width', 200); pd.set_option('display.max_columns', 60); pd.set_option('display.max_colwidth', 110)
from sklearn.model_selection import cross_val_predict
from sklearn.linear_model import LogisticRegression
from sklearn.dummy import DummyClassifier
from sklearn.pipeline import Pipeline
from sklearn.metrics import roc_auc_score, average_precision_score, precision_score, recall_score, f1_score, accuracy_score, confusion_matrix
from cortexhr import config as C, data_loading as D, audit as A, features as F, modeling as M, evaluation as E, reporting as R, runs, workflow as W
import cortexhr_helpers as H; H.DATA = ROOT / '03_Data' / 'csv'      # the OFFICIAL starter helper (unmodified): load, binary_report, risk_bands, catch_rate, group_gap, psi
print('Python', sys.version.split()[0], '| project root:', ROOT)

Python 3.14.8 | project root: C:\Users\USER\Downloads\cortexhr_phase1


In [2]:
from sklearn.ensemble import IsolationForest
from cortexhr import attendance as T

---
# Stage 1 of 12: Data Collection
Attendance sheets, the reference benchmark, and the vault status.

In [3]:
att = D.load_sheet(C.SHEET_ATT); daily = D.load_sheet('attendance_daily_examples'); bench = H.load('anomaly_baselines')
print('monthly:', att.shape, '| employees', att.employee_id.nunique(), '| daily labelled sample:', daily.shape)
labels_file = runs.find_attendance_labels()
print('hidden labels file in 04_Validation_Vault:', labels_file or 'NOT SUPPLIED yet - monthly precision/recall will be reported as NOT MEASURED')
bench

monthly: (7339, 16) | employees 528 | daily labelled sample: (2000, 13)
hidden labels file in 04_Validation_Vault: C:\Users\USER\Downloads\cortexhr_phase1\04_Validation_Vault\unlock_CX-05_attendance_anomaly_labels.csv


,task,features,precision,recall,flagged,injected
0,"Attendance anomalies (Isolation Forest, 3% contamination)","scheduled_days, present_days, half_days_unpaid, paid_leave_days, unpaid_absence_days, longest_unpaid_strea...",0.403,0.593,221,150
1,"Payroll anomalies (Isolation Forest on ratios, 2.5% contamination)","gross_vs_fixed, ded_ratio, net_ratio, overtime_hours_paid, dup_flag, net_pay",0.217,0.400,184,100


---
# Stage 2 of 12: Data Understanding
One row = one employee-month. What do the columns mean?

In [4]:
att.head(3).T

,0,1,2
attendance_feature_id,AF-00001,AF-00002,AF-00003
employee_id,EMP-0002,EMP-0002,EMP-0002
month,2025-04-01 00:00:00,2025-05-01 00:00:00,2025-06-01 00:00:00
department_id,D08,D08,D08
job_level,4,4,4
scheduled_days,20,21,21
present_days,20,20,19
wfh_days,18,18,18
half_days_unpaid,0,1,1
paid_leave_days,0,0,1


In [5]:
att.describe().T[['min', 'mean', 'max']].round(2)

,min,mean,max
month,2025-04-01 00:00:00,2025-12-21 12:49:56.157514,2026-09-01 00:00:00
job_level,1.0,2.721079,7.0
scheduled_days,1.0,20.459872,23.0
present_days,0.0,19.103284,23.0
wfh_days,0.0,8.041423,23.0
half_days_unpaid,0.0,0.089658,4.0
paid_leave_days,0.0,0.972203,23.0
unpaid_absence_days,0.0,0.294999,22.0
longest_unpaid_streak_days,0.0,0.263524,22.0
late_arrivals,0.0,1.369941,20.0


---
# Stage 3 of 12: Data Quality Audit
Audit of the attendance table (label-free).

In [6]:
chk = {'missing values': int(att.isna().sum().sum()), 'duplicate (employee, month)': int(att.duplicated(['employee_id', 'month']).sum()),
       'present_days > scheduled_days': int((att.present_days > att.scheduled_days).sum()), 'longest streak > unpaid absence days': int((att.longest_unpaid_streak_days > att.unpaid_absence_days).sum()),
       'wfh_days > present_days': int((att.wfh_days > att.present_days).sum()), 'late_arrivals > present_days': int((att.late_arrivals > att.present_days).sum()),
       'avg_hours_worked > 12': int((att.avg_hours_worked > 12).sum()), 'avg_hours < 3 while present': int(((att.avg_hours_worked < 3) & (att.present_days > 0)).sum()),
       'zero present days': int((att.present_days == 0).sum())}
pd.Series(chk, name='rows').to_frame()

,rows
missing values,0
"duplicate (employee, month)",0
present_days > scheduled_days,0
longest streak > unpaid absence days,0
wfh_days > present_days,19
late_arrivals > present_days,0
avg_hours_worked > 12,35
avg_hours < 3 while present,9
zero present days,50


---
# Stage 4 of 12: Exploratory Data Analysis (EDA)
How does attendance look across months and employees?

In [7]:
fig, ax = plt.subplots(1, 3, figsize=(14, 3.6))
att.groupby('month').attendance_rate.mean().plot(ax=ax[0], title='mean attendance rate by month')
ax[1].hist(att.avg_hours_worked, bins=50); ax[1].set_title('avg_hours_worked')
ax[2].hist(att.late_arrivals, bins=range(0, int(att.late_arrivals.max()) + 2)); ax[2].set_title('late_arrivals')
fig.tight_layout();

In [8]:
att[['attendance_rate', 'avg_hours_worked', 'late_arrivals', 'unpaid_absence_days', 'overtime_hours']].corr().round(2)

,attendance_rate,avg_hours_worked,late_arrivals,unpaid_absence_days,overtime_hours
attendance_rate,1.00,0.81,0.10,-0.50,0.03
avg_hours_worked,0.81,1.00,0.09,-0.27,0.13
late_arrivals,0.10,0.09,1.00,-0.03,-0.01
unpaid_absence_days,-0.50,-0.27,-0.03,1.00,-0.01
overtime_hours,0.03,0.13,-0.01,-0.01,1.00


---
# Stage 5 of 12: Data Cleaning
Cleaning policy: rows that break logic are NOT deleted, because unusual rows are exactly what we are hunting.

In [9]:
print('missing values to impute:', int(att.isna().sum().sum()), '| rows kept:', len(att))
print('Impossible/odd rows stay in the data and are surfaced by the rules and the model instead of being silently removed.')

missing values to impute: 0 | rows kept: 7339
Impossible/odd rows stay in the data and are surfaced by the rules and the model instead of being silently removed.


---
# Stage 6 of 12: Feature Engineering
Features: the attendance columns plus comparison with the same employee's own typical month.

In [10]:
features = ['scheduled_days', 'present_days', 'half_days_unpaid', 'paid_leave_days', 'unpaid_absence_days',
            'longest_unpaid_streak_days', 'late_arrivals', 'avg_hours_worked', 'overtime_hours', 'attendance_rate']
d = T.add_history_features(att); features_plus = features + T.HISTORY_FEATURES
T.assert_no_labels(features_plus); print('features:', features_plus)
d[T.HISTORY_FEATURES].describe().round(3)

features: ['scheduled_days', 'present_days', 'half_days_unpaid', 'paid_leave_days', 'unpaid_absence_days', 'longest_unpaid_streak_days', 'late_arrivals', 'avg_hours_worked', 'overtime_hours', 'attendance_rate', 'rate_vs_own_median', 'late_vs_own_median', 'hours_vs_own_median']


,rate_vs_own_median,late_vs_own_median,hours_vs_own_median
count,7339.000,7339.000,7339.000
mean,-0.039,0.209,-0.210
std,0.120,1.447,1.163
min,-1.000,-8.500,-8.920
25%,-0.050,-0.500,-0.370
50%,0.000,0.000,0.000
75%,0.003,1.000,0.220
max,0.236,20.000,10.190


---
# Stage 7 of 12: Train-Test Split
No labels exist for training, so the 'split' is by TIME: fit on the first 12 months, then score the later months to check the detector is stable.

In [11]:
months = sorted(d.month.unique()); cut = months[11]
fit_df, later = d[d.month <= cut], d[d.month > cut]
iso_t = IsolationForest(n_estimators=300, contamination=0.03, random_state=C.SEED).fit(fit_df[features_plus])
thr = np.quantile(-iso_t.score_samples(fit_df[features_plus]), 0.97)
print('fit months:', fit_df.month.min().date(), '->', cut.date(), '| later months:', later.month.min().date(), '->', later.month.max().date())
print(f'flag rate on later months using the early-months model: {100 * (-iso_t.score_samples(later[features_plus]) > thr).mean():.2f}% (target about 3%)')

fit months: 2025-04-01 -> 2026-03-01 | later months: 2026-04-01 -> 2026-09-01
flag rate on later months using the early-months model: 3.49% (target about 3%)


---
# Stage 8 of 12: Model Building
IsolationForest without labels (about 3% contamination) plus two simple rules, combined.

Isolation Forest builds random trees; an unusual row is isolated after only a few splits and gets a high score. *Contamination* = the share of rows we choose to flag.

In [12]:
iso = IsolationForest(n_estimators=300, contamination=0.03, random_state=C.SEED).fit(d[features_plus])
d['iforest_score'] = -iso.score_samples(d[features_plus]); d['iforest_flag'] = iso.predict(d[features_plus]) == -1
iso_raw = IsolationForest(n_estimators=300, contamination=0.03, random_state=C.SEED).fit(d[features]); d['iforest_raw_flag'] = iso_raw.predict(d[features]) == -1
print('model flags:', int(d.iforest_flag.sum()), 'of', len(d), '| raw-features-only variant flags', int(d.iforest_raw_flag.sum()))
d[d.iforest_flag].sort_values('iforest_score', ascending=False)[['attendance_feature_id', 'employee_id', 'month', 'present_days', 'attendance_rate', 'avg_hours_worked', 'late_arrivals', 'iforest_score']].head(8)

model flags: 221 of 7339 | raw-features-only variant flags 221


,attendance_feature_id,employee_id,month,present_days,attendance_rate,avg_hours_worked,late_arrivals,iforest_score
2818,AF-02819,EMP-0318,2026-06-01,0,0.0,0.0,0,0.769660
5874,AF-05875,EMP-0582,2026-01-01,0,0.0,0.0,0,0.768901
5917,AF-05918,EMP-0585,2025-09-01,0,0.0,0.0,0,0.767260
6634,AF-06635,EMP-0633,2026-08-01,0,0.0,0.0,0,0.766678
3169,AF-03170,EMP-0356,2025-07-01,0,0.0,0.0,0,0.766277
6442,AF-06443,EMP-0619,2026-08-01,0,0.0,0.0,0,0.765920
2086,AF-02087,EMP-0246,2025-08-01,0,0.0,0.0,0,0.765907
6801,AF-06802,EMP-0646,2026-06-01,0,0.0,0.0,0,0.765186


**Rules** (cheap, explainable, certain): R1 zero attendance in a whole month; R2 implausible hours (above 12 h/day, or below 3 h on days present).

In [13]:
d['rule_zero_attendance'] = (d.scheduled_days > 0) & (d.present_days == 0)
d['rule_implausible_hours'] = (d.avg_hours_worked > 12) | ((d.avg_hours_worked < 3) & (d.present_days > 0))
d['rule_flag'] = d.rule_zero_attendance | d.rule_implausible_hours; d['flagged'] = d.iforest_flag | d.rule_flag
print(d[['iforest_flag', 'rule_zero_attendance', 'rule_implausible_hours', 'rule_flag', 'flagged']].sum().to_frame('rows'))
print('flagged by both model and rules:', int((d.iforest_flag & d.rule_flag).sum()))

                        rows
iforest_flag             221
rule_zero_attendance      50
rule_implausible_hours    44
rule_flag                 94
flagged                  252
flagged by both model and rules: 63


### Think for yourself: if a detector flags 1,000 rows a month, who has time to check them? How to rank flags?

In [14]:
d['priority_score'] = d.iforest_score + d.rule_flag * 1.0       # rule-confirmed rows first, then by how unusual
queue = d[d.flagged].sort_values(['priority_score', 'attendance_feature_id'], ascending=[False, True]).reset_index(drop=True); queue.index += 1
queue[['attendance_feature_id', 'employee_id', 'month', 'rule_flag', 'iforest_score', 'priority_score']].head(10)

,attendance_feature_id,employee_id,month,rule_flag,iforest_score,priority_score
1,AF-02819,EMP-0318,2026-06-01,True,0.769660,1.769660
2,AF-05875,EMP-0582,2026-01-01,True,0.768901,1.768901
3,AF-05918,EMP-0585,2025-09-01,True,0.767260,1.767260
4,AF-06635,EMP-0633,2026-08-01,True,0.766678,1.766678
5,AF-03170,EMP-0356,2025-07-01,True,0.766277,1.766277
6,AF-06443,EMP-0619,2026-08-01,True,0.765920,1.765920
7,AF-02087,EMP-0246,2025-08-01,True,0.765907,1.765907
8,AF-06802,EMP-0646,2026-06-01,True,0.765186,1.765186
9,AF-05661,EMP-0570,2025-11-01,True,0.764233,1.764233
10,AF-03112,EMP-0347,2026-09-01,True,0.763177,1.763177


**Answer.** Rank by `priority_score`: rule-confirmed rows first (explainable and quick to verify), then the rest by anomaly score. Reviewers work top-down within monthly capacity (e.g. top 100). Also suppress repeat flags for the same employee, and add leave status so legitimate long absences (e.g. maternity leave, a known false-positive trap) are not escalated.

---
# Stage 9 of 12: Model Evaluation
Flags are saved FIRST. THEN labels are opened and used only to measure (catch_rate). Confusion matrix included.

In [15]:
flags_out = d.loc[d.flagged, ['attendance_feature_id', 'employee_id', 'month', 'iforest_score', 'priority_score']]
flags_out.to_csv(ROOT / 'reports' / 'tables' / 'attendance_flags_notebook.csv', index=False)    # written before any label is read
print('saved', len(flags_out), 'flags before reading labels')
print(f'benchmark to compare: precision {bench.iloc[0].precision}, recall {bench.iloc[0].recall}, flagged {int(bench.iloc[0].flagged)}, injected {int(bench.iloc[0].injected)}')

saved 252 flags before reading labels
benchmark to compare: precision 0.403, recall 0.593, flagged 221, injected 150


In [16]:
if labels_file is not None:
    labels = T.load_labels(labels_file); key = 'attendance_feature_id'; truth = set(labels[key]); rows = []
    for name, mask in {'IsolationForest (raw features)': d.iforest_raw_flag, 'IsolationForest (+history)': d.iforest_flag, 'Rules only': d.rule_flag, 'Combined': d.flagged}.items():
        r = H.catch_rate(d.loc[mask, key], truth); r['detector'] = name; rows.append(r)       # OFFICIAL catch_rate
    measured = pd.DataFrame(rows)[['detector', 'flagged', 'real', 'caught', 'precision', 'recall']].round(3); show = measured
    y_true = d[key].isin(truth); tp = int((d.flagged & y_true).sum()); fp = int((d.flagged & ~y_true).sum()); fn = int((~d.flagged & y_true).sum()); tn = int((~d.flagged & ~y_true).sum())
    W.plot_confusion_counts(tn, fp, fn, tp, 'Monthly combined detector vs hidden labels')
else:
    measured = None; show = 'NOT MEASURED: labels file missing. Put it in 04_Validation_Vault/ and re-run this notebook.'
show

,detector,flagged,real,caught,precision,recall
0,IsolationForest (raw features),221,150,88,0.398,0.587
1,IsolationForest (+history),221,150,95,0.430,0.633
2,Rules only,94,150,50,0.532,0.333
3,Combined,252,150,126,0.500,0.840


**Which anomaly types are hardest?** (needs the labels file)

In [17]:
if labels_file is not None:
    tcol = next(c for c in labels.columns if 'type' in c.lower()); caught = set(d.loc[d.flagged, key])
    by_type = labels.assign(caught=labels[key].isin(caught)).groupby(tcol).agg(injected=('caught', 'size'), caught=('caught', 'sum'))
    by_type['missed'] = by_type.injected - by_type.caught; by_type['recall'] = (by_type.caught / by_type.injected).round(3); out = by_type
else:
    out = 'Needs the labels file (anomaly_type column).'
out

,injected,caught,missed,recall
anomaly_type,,,,
ABSENCE_STREAK,40,40,0,1.0
IMPOSSIBLE_HOURS,35,35,0,1.0
LATE_SPIKE,30,6,24,0.2
SUDDEN_DROP,30,30,0,1.0
ZERO_ATTENDANCE,15,15,0,1.0


**My note on the hard types (measured on the hidden labels; rewrite it in your own words for your demo).**
- **Hardest: `LATE_SPIKE`** (recall 0.20, 24 of 30 missed). A spike of 11-20 late arrivals is an extreme in *one* column, but the Isolation Forest scores *combinations* of columns; with only 3% of rows to flag, single-column spikes were out-ranked by rows that were odd in several columns, and none of my rules covered late arrivals.
- **Easy: `IMPOSSIBLE_HOURS`** is caught entirely by the hours rule (model alone only 11%), `ABSENCE_STREAK`, `SUDDEN_DROP` and `ZERO_ATTENDANCE` entirely by the model/own-history features.
- **False alarms:** about half of all flags are not injected anomalies. 35 came from the zero-attendance rule on months fully covered by paid leave (legitimate, e.g. maternity): the known false-positive trap, fixed only by using leave status.
- The v1 detector above was fixed BEFORE the labels were opened (flags committed first). The 'post-hoc v2' row below is written AFTER seeing this table, so it is optimistic and must not be presented as an honest test.

In [18]:
if labels_file is not None:
    ok_leave = d.paid_leave_days >= 0.8 * d.scheduled_days                       # post-hoc: months fully covered by paid leave are legitimate
    r1b = d.rule_zero_attendance & ~ok_leave; r3 = d.late_arrivals > d.late_arrivals.quantile(0.995)
    v2 = d.iforest_flag | r1b | d.rule_implausible_hours | r3
    print('v1 (honest) combined :', {k: (round(v, 3) if isinstance(v, float) else v) for k, v in H.catch_rate(d.loc[d.flagged, key], truth).items()})
    print('POST-HOC v2 (optimistic, written after seeing the labels):', {k: (round(v, 3) if isinstance(v, float) else v) for k, v in H.catch_rate(d.loc[v2, key], truth).items()})
else: print('needs the labels file')

v1 (honest) combined : {'flagged': 252, 'real': 150, 'caught': 126, 'precision': 0.5, 'recall': 0.84}
POST-HOC v2 (optimistic, written after seeing the labels): {'flagged': 278, 'real': 150, 'caught': 150, 'precision': 0.54, 'recall': 1.0}


**Supplementary measured check on the labelled DAILY sample** (a different table shipped in the main workbook; not a substitute for the monthly measurement).

In [19]:
dp = T.daily_proxy_eval(daily); print(dp.round(3).to_string(index=False))
row = dp[dp.detector == 'Daily combined'].iloc[0]; tp, fp, fn = int(row.caught), int(row.flagged - row.caught), int(row.injected - row.caught)
W.plot_confusion_counts(len(daily) - tp - fp - fn, fp, fn, tp, 'Daily sample: combined detector');

                detector  flagged  injected  caught  precision  recall
   Daily IsolationForest       60       120      47      0.783   0.392
Daily rule (hours/clock)       93       120      90      0.968   0.750
          Daily combined      107       120      93      0.869   0.775


---
# Stage 10 of 12: Hyperparameter Tuning
Sensitivity of the contamination setting. The task fixes it at about 3%; this only shows how flag volume and (daily-sample) catch rate change.

In [20]:
rows = []
for c in [0.01, 0.02, 0.03, 0.05]:
    f = IsolationForest(n_estimators=300, contamination=c, random_state=C.SEED).fit(d[features_plus]); n_flag = int((f.predict(d[features_plus]) == -1).sum())
    dpc = T.daily_proxy_eval(daily, contamination=c).set_index('detector').loc['Daily IsolationForest']
    rows.append({'contamination': c, 'monthly rows flagged by model': n_flag, 'daily-sample precision': round(dpc.precision, 3), 'daily-sample recall': round(dpc.recall, 3)})
pd.DataFrame(rows)

,contamination,monthly rows flagged by model,daily-sample precision,daily-sample recall
0,0.01,74,1.000,0.108
1,0.02,147,0.900,0.300
2,0.03,221,0.783,0.392
3,0.05,367,0.760,0.633


The final setting stays at 3% (as the task specifies), not chosen from labels. A higher contamination catches more but floods reviewers, which is the 1,000-flags problem.

---
# Stage 11 of 12: Model Interpretation
Why was a row flagged? Which features separate flagged rows from normal ones?

In [21]:
comp = d.groupby('flagged')[features_plus].mean().T.rename(columns={False: 'normal', True: 'flagged'}); comp['ratio'] = (comp.flagged / comp.normal.replace(0, np.nan)).round(2)
comp.sort_values('ratio', key=lambda s: (s - 1).abs(), ascending=False).round(3).head(10)

flagged,normal,flagged,ratio
unpaid_absence_days,0.170,3.817,22.49
longest_unpaid_streak_days,0.155,3.306,21.28
rate_vs_own_median,-0.024,-0.449,18.44
hours_vs_own_median,-0.155,-1.770,11.42
paid_leave_days,0.800,5.802,7.25
half_days_unpaid,0.078,0.429,5.52
present_days,19.450,9.341,0.48
attendance_rate,0.951,0.502,0.53
avg_hours_worked,8.389,6.614,0.79
late_vs_own_median,0.209,0.228,1.09


In [22]:
row = d[d.flagged].sort_values('priority_score', ascending=False).iloc[0]
print('Top-priority row', row.attendance_feature_id, row.employee_id, row.month.date(), '| reason:',
      'zero attendance' if row.rule_zero_attendance else 'implausible hours' if row.rule_implausible_hours else 'unusual combination')
print((row[features_plus] - d[features_plus].median()).round(2).sort_values(key=abs, ascending=False).head(6).to_string())

Top-priority row AF-02819 EMP-0318 2026-06-01 | reason: zero attendance
unpaid_absence_days           22.0
longest_unpaid_streak_days    22.0
present_days                 -20.0
hours_vs_own_median           -8.8
avg_hours_worked             -8.59
scheduled_days                 2.0


---
# Stage 12 of 12: Final Conclusions and Recommendations
Conclusions and recommendations.

In [23]:
print(f'''DETECTOR: IsolationForest (3% contamination) + 2 rules flags {int(d.flagged.sum())} of {len(d)} employee-months ({100 * d.flagged.mean():.1f}%); rules cover obvious cases, the model covers unusual patterns.
MEASUREMENT: {'monthly labels (measurement only): combined precision ' + format(measured.set_index('detector').loc['Combined'].precision, '.3f') + ', recall ' + format(measured.set_index('detector').loc['Combined'].recall, '.3f') + ' vs benchmark ' + format(bench.iloc[0].precision, '.3f') + ' / ' + format(bench.iloc[0].recall, '.3f') if labels_file is not None else 'monthly precision/recall NOT measured (labels file not supplied); daily-sample check: combined precision ' + format(dp.set_index('detector').loc['Daily combined'].precision, '.2f') + ', recall ' + format(dp.set_index('detector').loc['Daily combined'].recall, '.2f')}.
RECOMMENDATIONS: 1) rank flags by priority and review the top N; 2) add leave status to avoid flagging legitimate long absences; 3) measure monthly precision/recall as soon as the hidden labels are available; 4) reuse this method for payroll in Phase 2.''')

DETECTOR: IsolationForest (3% contamination) + 2 rules flags 252 of 7339 employee-months (3.4%); rules cover obvious cases, the model covers unusual patterns.
MEASUREMENT: monthly labels (measurement only): combined precision 0.500, recall 0.840 vs benchmark 0.403 / 0.593.
RECOMMENDATIONS: 1) rank flags by priority and review the top N; 2) add leave status to avoid flagging legitimate long absences; 3) measure monthly precision/recall as soon as the hidden labels are available; 4) reuse this method for payroll in Phase 2.


### Completion checklist (checked by code)

In [24]:
checks = {
    "'You never trained on the labels (label-like columns blocked; flags saved before labels)'": True,
    "'Rules and model are both explained and combined'": int(d.flagged.sum()) >= int(d.iforest_flag.sum()),
    "'Contamination is about 3%'": abs(d.iforest_flag.mean() - 0.03) < 0.005,
    "'Precision and recall reported against hidden labels (needs the labels file)'": True if labels_file is not None else 'MANUAL',
    "'Note on the anomaly types that were hard (needs the labels file)'": True if labels_file is not None else 'MANUAL'
}
status = {k: ('MANUAL' if isinstance(v, str) and v == 'MANUAL' else 'PASS' if bool(v) else 'FAIL') for k, v in checks.items()}
for k, v in status.items(): print(f'[{v}]', k)
assert 'FAIL' not in status.values()

[PASS] 'You never trained on the labels (label-like columns blocked; flags saved before labels)'
[PASS] 'Rules and model are both explained and combined'
[PASS] 'Contamination is about 3%'
[PASS] 'Precision and recall reported against hidden labels (needs the labels file)'
[PASS] 'Note on the anomaly types that were hard (needs the labels file)'


In [25]:
res5 = runs.run_attendance(write=True)    # regenerates tables, figure and reports/attendance_anomaly_report.md
print('report written:', (C.reports_dir() / 'attendance_anomaly_report.md').exists())

report written: True


### Evidence for your report
Metrics table (`reports/tables/attendance_detector_metrics.csv` once labels exist) and the list of missed types with your idea why.
References: [scikit-learn](https://github.com/scikit-learn/scikit-learn) | [pandas](https://github.com/pandas-dev/pandas) | [PyOD](https://github.com/yzhao062/pyod)
**How real companies do it:** labels are for measuring, never for training an unsupervised detector; mixing them breaks the measurement.
**Connects to Phase 2:** the same method for payroll, where mistakes cost real money.
**You can now say:** *I built and measured an anomaly detector with no labels.*

## A5. `05_attendance_anomalies.ipynb` (Task 5)

### Stage 8 — Think for yourself: if a detector flags 1,000 rows a month, who has time to check them? How to rank the flags?

**Answer.** Nobody can check 1,000 rows, so the detector must hand over a ranked list. I rank by `priority_score` = the Isolation Forest anomaly score plus 1.0 if an explainable rule fired. Rule-confirmed rows (zero attendance, impossible hours) go first because they are certain and quick to verify; the rest follow by how unusual they are. Reviewers work from the top within their monthly capacity (for example the top 100). Two more ideas: suppress repeat flags for the same employee, and use leave status so legitimate long absences (maternity, long paid leave) are not escalated.

### Stage 9 — Measurement table

**Result against the hidden labels (flags were saved and committed before the labels were opened).** Combined detector: 252 flagged, 126 of 150 real anomalies caught: precision 0.500, recall 0.840, against the benchmark 0.403 / 0.593. The model alone with own-history features: 0.430 / 0.633; rules alone: 0.532 / 0.333. Rules and model complement each other.

### Stage 9 — By-type table: which anomaly types are hardest and why?

**Hardest: `LATE_SPIKE`.** Only 6 of 30 caught (recall 0.20). A spike of 11-20 late arrivals is extreme in *one* column, but Isolation Forest scores *combinations* of columns; with only 3% of rows to flag, single-column spikes were out-ranked by rows that were odd in several columns, and none of my rules covered late arrivals. **Easy:** `IMPOSSIBLE_HOURS` (35/35, found by the hours rule; the model alone finds only 11%), `ABSENCE_STREAK` (40/40) and `SUDDEN_DROP` (30/30) (found by the model with own-history features; no rule covers them) and `ZERO_ATTENDANCE` (15/15, found by both the rule and the model).

**False alarms:** about half of the flags (126 of 252) were not injected anomalies; 35 came from the zero-attendance rule on months fully covered by paid leave (legitimate, e.g. maternity): the known false-positive trap, fixed only by using leave status.

**Post-hoc v2** (excluding paid-leave months and flagging late arrivals above the 99.5th percentile) reaches precision 0.54 / recall 1.00, but I wrote it after seeing the labels, so it is optimistic and I do not present it as an honest test.

### Stages 7 and 10 — Stability, contamination

**Stability.** Fitting on the first 12 months and scoring the later 6 flags 3.49% of rows (target 3%), so the detector is stable over time. **Contamination:** 1% flags 74 monthly rows, 3% flags 221, 5% flags 367. On the labelled daily sample precision falls from 1.00 to 0.76 as recall rises from 0.11 to 0.63: higher contamination catches more but floods reviewers. I keep the task's 3%.

### Stage 12 — Conclusions

**Conclusions.** The Isolation Forest (3% contamination) plus two simple rules flags 252 of 7,339 employee-months (3.4%) and beats the benchmark on both precision (0.50 vs 0.40) and recall (0.84 vs 0.59). Labels were used only to measure. **Recommendations:** review flags by priority; add leave status; add a rule or feature for late-arrival spikes; reuse the method for payroll in Phase 2.